In [1]:
import numpy as np
import torch
x = ((np.random.randn(4), np.random.randn(6), np.random.randn(3, 4, 4)), (np.random.randn(4), np.random.randn(6), np.random.randn(3, 4, 4)), (np.random.randn(4), np.random.randn(6), np.random.randn(3, 4, 4)), (np.random.randn(4), np.random.randn(6), np.random.randn(3, 4, 4)))

In [4]:
%%timeit
y = zip(*x)
unpacked_tuples = tuple(map(tuple, y))
list = []
for t in unpacked_tuples:
    list.append(np.stack(t))

final1D, final3D = np.concatenate(list[:2], -1), list[-1]
final1D.shape, final3D.shape

13.9 µs ± 28.5 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [6]:
%%timeit
allObs1D, allObs3D = [], []
for observation in x:
    # print(f"processObservations: observation: {observation}")
    obs1D, obs3D = [], []
    for observationElement in observation:
        # print(f"processObservations: observationElement: {observationElement}")
        observationElement = torch.tensor(observationElement)
        if observationElement.ndim == 1:
            obs1D.append(observationElement)
        elif observationElement.ndim == 3:
            obs3D.append(observationElement)
        else:
            print(f"Unexpected {observationElement.ndim}-dimensional observation")
    if obs1D:
        allObs1D.append(torch.cat(obs1D))
    if obs3D:
        allObs3D.append(torch.cat(obs3D))

final1D = torch.stack(allObs1D) if allObs1D else torch.empty(0, dtype=torch.float32)
final3D = torch.stack(allObs3D) if allObs3D else torch.empty(0, dtype=torch.float32)
final1D, final3D

92.6 µs ± 621 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [7]:
%%timeit
observationElements = tuple(map(tuple, zip(*x)))
obs1D, obs3D = [], []
for observationElement in observationElements:
    if observationElement[0].ndim == 1:
        obs1D.append(torch.from_numpy(np.stack(observationElement)))
    elif observationElement[0].ndim == 3:
        obs3D.append(torch.from_numpy(np.stack(observationElement)))
    else:
        print(f"Unexpected {observationElement[0].ndim}-dimensional observation")

final1D = torch.cat(obs1D, -1) if obs1D else torch.empty(0, dtype=torch.float32)
final3D = torch.cat(obs3D, -1) if obs3D else torch.empty(0, dtype=torch.float32)
final1D, final3D

24.8 µs ± 306 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [2]:
dict1 = {"a":1, "b":2, "c":3}
dict2 = {"d":4, "e":5, "f":6}
dict3 = dict1 | dict2
dict3

{'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6}

In [13]:
observationElements[0][0].ndim

1

In [16]:
for observationElement in observationElements:
    print(f"Element: {observationElement}")
    print(f"First subelement to check dimensionality: {observationElement[0]}")
    print(f"Subelements dimensionality: {observationElement[0].ndim}")
    

Element: (array([ 0.9113481 , -0.85951092, -1.50139671,  0.59073771]), array([-0.24068528,  0.28120071,  1.15998356, -2.15282646]), array([0.35300083, 0.45937574, 0.81385764, 0.04044255]), array([-0.99056256, -1.00506618,  0.76477579, -1.47700431]))
First subelement to check dimensionality: [ 0.9113481  -0.85951092 -1.50139671  0.59073771]
Subelements dimensionality: 1
Element: (array([-0.13782244,  0.15113564, -0.0876594 ,  0.82729982,  0.9243366 ,
        1.59506249]), array([ 2.01821727, -0.23033904,  0.64216612,  0.02877579,  0.6763248 ,
        0.17735145]), array([ 1.81575058,  0.02615307, -1.3721234 , -0.34392826,  0.27801123,
       -1.18269463]), array([-0.87094772, -1.28570354, -1.08497408,  0.23409723,  0.35079344,
        0.57804679]))
First subelement to check dimensionality: [-0.13782244  0.15113564 -0.0876594   0.82729982  0.9243366   1.59506249]
Subelements dimensionality: 1
Element: (array([[[-0.93264044, -0.24130747,  0.91593562,  0.25834448],
        [ 0.75295908,  0